# Sensitivity analysis on the availability of carbon capture and storage

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from utils import *
import bw2data as bd
import pickle
from new_plots import _create_sankey_figure, plot_sankey, generate_sankey_flows

In [ ]:
save_results = True

In [ ]:
year = 2050
path_data = f'../02_AMPL_files/data/{year}/'

In [ ]:
es_tech_df = pd.read_csv('../01_Notebooks/Data/technology_dictionary.csv')
model = pd.read_csv(f'../01_Notebooks/Data/model_{year}.csv')

In [ ]:
es_tech_df = es_tech_df[~es_tech_df['Programming name'].isin(wood_list+wet_biomass_list+waste_list)]  # keeping ES names for aggregation of biomass resources later on
es_tech_name_dict = dict(zip(es_tech_df['Programming name'], es_tech_df['Long name']))

In [ ]:
for distance_level in ['_SD', '_MD', '_LD', '_ELD']:
    model['Name'] = model['Name'].str.replace(distance_level, '')
    model['Flow'] = model['Flow'].str.replace(distance_level, '')
model.drop_duplicates(inplace=True)

model['Name'] = model['Name'].replace(es_tech_name_dict)

## Define the sequence

In [ ]:
avail_max = 36
avail_min = 7
N_run = int((avail_max - avail_min) / 0.5 + 1)

avail = [
    {
        'name': 'seq_limit_max',
        'lower_bound': avail_min * 1000,
        'upper_bound': avail_max * 1000,
    },
]
data = []
for i in range(len(avail)):
    data.append(['seq_limit_max', None, None, None, None] + list(np.linspace(avail[i]['lower_bound'], avail[i]['upper_bound'], N_run)))

columns = ['param', 'index0', 'index1', 'index2', 'index3'] + [f'value{i+1}' for i in range(N_run)]
seq_data = pd.DataFrame(data)
seq_data.columns = columns

## Run the model

In [ ]:
ssp_rcp_list = ['SSP2-L', 'SSP5-H']
reg_level_list = ['base_wo_iam', 'base', 'spat', 'spat_fore', 'spat_back', 'spat_fore_back']

In [ ]:
for ssp_rcp in ssp_rcp_list:

    for reg_level in reg_level_list:

        if ssp_rcp == 'SSP2-L' and reg_level == 'base_wo_iam':
            continue

        es = run_opti(
            reg_level=reg_level,
            year=year,
            ssp_rcp=ssp_rcp,
            other_emissions=True,
            constraint_on_remaining_aop=True,
            constraint_on_foreign_ghg_emissions=True,
            returns='model',
        )

        # Solve the model and get results
        res = es.calc_sequence(seq_data)
        res = filter_numerical_errors(res)
        res = collapse_temporal_index(res)
        res = postprocessing(res)

        if save_results:
            with open(f'../03_Results/Tables/sensitivity_analysis/{reg_level}/{ssp_rcp}/results.pickle', 'wb') as f:
                pickle.dump(res, f)

        print(f"{ssp_rcp}-{reg_level} done!")

In [ ]:
for ssp_rcp in ssp_rcp_list:

    for reg_level in reg_level_list:

        if ssp_rcp == 'SSP2-L' and reg_level == 'base_wo_iam':
            continue

        with open(f'../03_Results/Tables/sensitivity_analysis/{reg_level}/{ssp_rcp}/results.pickle', 'rb') as f:
            res = pickle.load(f)

        for i in range(1, N_run+1):

            fig_data = generate_sankey_flows(
                results=res,
                aggregate_mobility=True,
                aggregate_grid=True,
                aggregate_technology=True,
                run_id=i,
            )
            fig_data['source (long)'] = fig_data.apply(lambda x: es_tech_name_dict[x['source']] if x['source'] in es_tech_name_dict else x['source'], axis=1)
            fig_data['target (long)'] = fig_data.apply(lambda x: es_tech_name_dict[x['target']] if x['target'] in es_tech_name_dict else x['target'], axis=1)

            fig = _create_sankey_figure(fig_data, colors=default_colors_sankey, long_names=True)
            if save_results:
                fig.write_html(f'../03_Results/Figures/sensitivity_analysis/{reg_level}/{ssp_rcp}/sankey_{i}.html')

## Carbon sankey

In [ ]:
bd.projects.set_current('ecoinvent3.10.1')

In [ ]:
impact_categories_list = [
    i for i in bd.methods if
    (i[0] == 'IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10')
    | (i[0] == 'IMPACT World+ Midpoint 2.1_regionalized for ecoinvent v3.10')
    | (i[0] == 'IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)')
    | (i[0] == 'IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)')
]

impact_categories_list +=[
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Remaining human health'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10', 'Human health', 'Total human health (biogenic)'),
]

In [ ]:
for ssp_rcp in ssp_rcp_list:

    for reg_level in reg_level_list:

        if ssp_rcp == 'SSP2-L' and reg_level == 'base_wo_iam':
            continue

        impact_scores_2023 = pd.read_csv(f'../03_Results/LCA/2023/{"base" if reg_level == "base_wo_iam" else reg_level}/impact_scores.csv')
        impact_scores = pd.read_csv(f'../03_Results/LCA/2050/{reg_level}/{ssp_rcp}/impact_scores.csv')
        impact_scores_direct = pd.read_csv(f'../03_Results/LCA/2050/{reg_level}/{ssp_rcp}/impact_scores_direct_emissions.csv')
        impact_abbrev = pd.read_csv('../01_Notebooks/Data/impact_abbrev.csv')

        impact_scores = update_existing_infrastructure_metrics(
            df_impact_2050=impact_scores,
            df_impact_2020=impact_scores_2023,
            list_existing_techs=['HYDRO_DAM', 'HYDRO_RIVER', 'WIND_ONSHORE'],
        )

        impact_scores, impact_abbrev = add_rhhd_and_reqd_to_impact_scores_df(impact_scores, impact_abbrev)
        impact_scores, impact_abbrev = add_biogenic_climate_change_to_impact_scores_df(impact_scores, impact_abbrev)

        impact_scores_direct = add_rhhd_and_reqd_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]
        impact_scores_direct = add_biogenic_climate_change_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]

        with open(f'../03_Results/Tables/sensitivity_analysis/{reg_level}/{ssp_rcp}/results.pickle', 'rb') as f:
            res = pickle.load(f)

        df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores,
            df_results=res,
        )

        df_annual_prod_direct = get_impact_scores(
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores_direct,
            df_results=res,
            assessment_type='direct',
        )

        df_f_mult = df_f_mult[df_f_mult.F_Mult != 0]
        df_annual_prod = df_annual_prod[df_annual_prod.Annual_Prod != 0]
        df_annual_prod_direct = df_annual_prod_direct[df_annual_prod_direct.Annual_Prod != 0]
        df_annual_res = df_annual_res[df_annual_res.Annual_Res != 0]

        df_annual_prod['Sector'] = df_annual_prod.apply(category_to_sector, axis=1)
        df_annual_prod_direct['Sector'] = df_annual_prod_direct.apply(category_to_sector, axis=1)
        df_f_mult['Sector'] = df_f_mult.apply(category_to_sector, axis=1)
        df_annual_res['Sector'] = df_annual_res.apply(lambda x: 'Electricity' if x['index'] == 'ELECTRICITY_EHV' else 'Energy resources (excl. electricity)', axis=1)
        df_annual_res['Category'] = df_annual_res.apply(lambda x: 'ELECTRICITY_EHV' if x['index'] == 'ELECTRICITY_EHV' else 'Energy resources (excl. electricity)', axis=1)

        df_annual_prod = aggregate_mobility_submodels(df_annual_prod)
        df_annual_prod_direct = aggregate_mobility_submodels(df_annual_prod_direct)
        df_f_mult = aggregate_mobility_submodels(df_f_mult)

        df_f_mult.dropna(inplace=True)
        df_annual_prod.dropna(inplace=True)
        df_annual_prod_direct.dropna(inplace=True)
        df_annual_res.dropna(inplace=True)

        df_annual_prod = pd.merge(df_annual_prod, res.postprocessing['df_annual'].reset_index()[['index', 'Run', 'C_inv_an']], how='left', on=['index', 'Run'])

        # df_annual_res = df_annual_res.apply(lambda row: rename_bio_resources(row), axis=1)

        df_annual_prod['index'] = df_annual_prod['index'].replace(es_tech_name_dict)
        df_annual_prod_direct['index'] = df_annual_prod_direct['index'].replace(es_tech_name_dict)
        df_f_mult['index'] = df_f_mult['index'].replace(es_tech_name_dict)
        df_annual_res['index'] = df_annual_res['index'].replace(es_tech_name_dict)

        df_annual_prod_indirect = df_annual_prod.merge(df_annual_prod_direct, on=['index', 'Run', 'Sector', 'Category', 'Annual_Prod'], suffixes=('', ' (direct)'), how='left')
        for col in list(df_annual_prod.columns):
            if col not in ['index', 'Run', 'Sector', 'Category', 'Annual_Prod', 'Phase', 'C_inv_an']:
                df_annual_prod_indirect[col] = df_annual_prod_indirect[col] - df_annual_prod_indirect[f"{col} (direct)"]
                df_annual_prod_indirect.drop(columns=[f"{col} (direct)"], inplace=True)

        cat_list = ['Climate change, short term, total']
        col = ['Run', 'index', 'Sector', 'Phase'] + cat_list
        df_f_mult['Phase'] = 'Construction'
        df_annual_prod['Phase'] = 'Operation'
        df_annual_prod_direct['Phase'] = 'Operation (direct)'
        df_annual_prod_indirect['Phase'] = 'Operation (indirect)'
        df_annual_res['Phase'] = 'Resource'
        df_total_impact = pd.concat([
            df_f_mult[['F_Mult'] + col].rename(columns={'F_Mult': 'Capacity or production'}),
            df_annual_prod_direct[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
            df_annual_prod_indirect[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
            df_annual_res[['Annual_Res'] + col].rename(columns={'Annual_Res': 'Capacity or production'}),
        ],
            ignore_index=True)

        df_total_impact['Climate change, short term, total'] *= 1e3 / N_capita_2050

        for i in range(1, N_run+1):

            fig = plot_sankey_carbon_flows(
                run=i,
                df_total_impact=df_total_impact,
                model=model,
                cutoff=0,
                aggregate_technologies=True,
                show_figure=False,
                save_results=False,
                per_capita=False,
                mode='mfa',
                return_df=False,
                return_figure=True,
            )
            if save_results:
                fig.write_html(f'../03_Results/Figures/sensitivity_analysis/{reg_level}/{ssp_rcp}/sankey_carbon_{i}.html')

## Visualize results

In [ ]:
list_df_annual_prod = []
list_df_annual_res = []
list_df_f_mult = []
list_df_total_cost = []

for ssp_rcp in ssp_rcp_list:

    for reg_level in reg_level_list:

        if ssp_rcp == 'SSP2-L' and reg_level == 'base_wo_iam':
            continue

        with open(f'../03_Results/Tables/sensitivity_analysis/{reg_level}/{ssp_rcp}/results.pickle', 'rb') as f:
            res = pickle.load(f)

        impact_scores_2023 = pd.read_csv(f'../03_Results/LCA/2023/{"base" if reg_level == "base_wo_iam" else reg_level}/impact_scores.csv')
        impact_scores = pd.read_csv(f'../03_Results/LCA/2050/{reg_level}/{ssp_rcp}/impact_scores.csv')
        impact_scores_direct = pd.read_csv(f'../03_Results/LCA/2050/{reg_level}/{ssp_rcp}/impact_scores_direct_emissions.csv')
        impact_abbrev = pd.read_csv('../01_Notebooks/Data/impact_abbrev.csv')

        impact_scores = update_existing_infrastructure_metrics(
            df_impact_2050=impact_scores,
            df_impact_2020=impact_scores_2023,
            list_existing_techs=['HYDRO_DAM', 'HYDRO_RIVER', 'WIND_ONSHORE'],
        )

        impact_scores, impact_abbrev = add_rhhd_and_reqd_to_impact_scores_df(impact_scores, impact_abbrev)
        impact_scores, impact_abbrev = add_biogenic_climate_change_to_impact_scores_df(impact_scores, impact_abbrev)

        impact_scores_direct = add_rhhd_and_reqd_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]
        impact_scores_direct = add_biogenic_climate_change_to_impact_scores_df(impact_scores_direct, impact_abbrev)[0]

        df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
            impact_category=impact_categories_list,
            df_impact_scores=impact_scores,
            df_results=res,
        )

        df_total_cost = res.variables['TotalCost']

        df_annual_prod['SSP-RCP'] = ssp_rcp if reg_level != 'base_wo_iam' else 'base_wo_iam'
        df_f_mult['SSP-RCP'] = ssp_rcp if reg_level != 'base_wo_iam' else 'base_wo_iam'
        df_annual_res['SSP-RCP'] = ssp_rcp if reg_level != 'base_wo_iam' else 'base_wo_iam'
        df_total_cost['SSP-RCP'] = ssp_rcp if reg_level != 'base_wo_iam' else 'base_wo_iam'

        df_annual_prod['Regionalization level'] = reg_level
        df_f_mult['Regionalization level'] = reg_level
        df_annual_res['Regionalization level'] = reg_level
        df_total_cost['Regionalization level'] = reg_level

        list_df_annual_prod.append(df_annual_prod)
        list_df_annual_res.append(df_annual_res)
        list_df_f_mult.append(df_f_mult)
        list_df_total_cost.append(df_total_cost)

df_annual_prod = pd.concat(list_df_annual_prod)
df_f_mult = pd.concat(list_df_f_mult)
df_annual_res = pd.concat(list_df_annual_res)
df_total_cost = pd.concat(list_df_total_cost)

df_annual_prod['Sector'] = df_annual_prod.apply(category_to_sector, axis=1)
df_f_mult['Sector'] = df_f_mult.apply(category_to_sector, axis=1)
df_annual_res['Sector'] = df_annual_res.apply(lambda x: 'Electricity' if x['index'] == 'ELECTRICITY_EHV' else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'), axis=1)

In [ ]:
df_annual_prod['Run'] = df_annual_prod['Run'].apply(lambda x: np.linspace(avail[0]['lower_bound'], avail[0]['upper_bound'], N_run)[x-1])
df_annual_res['Run'] = df_annual_res['Run'].apply(lambda x: np.linspace(avail[0]['lower_bound'], avail[0]['upper_bound'], N_run)[x-1])
df_f_mult['Run'] = df_f_mult['Run'].apply(lambda x: np.linspace(avail[0]['lower_bound'], avail[0]['upper_bound'], N_run)[x-1])
df_total_cost['Run'] = df_total_cost['Run'].apply(lambda x: np.linspace(avail[0]['lower_bound'], avail[0]['upper_bound'], N_run)[x-1])

In [ ]:
df_annual_prod['Annual_Prod'] *= 1e-3 # from GWh to TWh
df_annual_res['Annual_Res'] *= 1e-3
df_annual_prod['Run'] *= 1e-3 # from kt CO2 to Mt CO2
df_annual_res['Run'] *= 1e-3
df_f_mult['Run'] *= 1e-3
df_total_cost['Run'] *= 1e-3

In [ ]:
reg_colors = {
    "base_wo_iam":   "black",
    "base":          "#185FA5",  # blue
    "spat":          "#1D9E75",  # teal
    "spat_fore":     "#BA7517",  # amber
    "spat_back":     "#993C1D",  # coral/burnt orange
    "spat_fore_back":"#7F77DD",  # purple
}

ssp_ls = {
    "SSP2-L": "solid",
    "SSP5-H": "dashdot",
    "base_wo_iam": "dot",
}

c = {
    "SSP2-L": "blue",
    "SSP5-H": "orange",
    "base_wo_iam": "black",
}

In [ ]:
# Infeasible runs are removed from plots
df_annual_prod_grouped = df_annual_prod.groupby(['SSP-RCP', 'Regionalization level', 'Run']).sum()[['Annual_Prod']]
infeasible_runs = df_annual_prod_grouped[df_annual_prod_grouped['Annual_Prod'] == 0].index.tolist()
df_annual_prod['Configuration'] = list(zip(
    df_annual_prod['SSP-RCP'],
    df_annual_prod['Regionalization level'],
    df_annual_prod['Run']
))
df_annual_prod = df_annual_prod[~df_annual_prod['Configuration'].isin(infeasible_runs)]
df_f_mult['Configuration'] = list(zip(
    df_f_mult['SSP-RCP'],
    df_f_mult['Regionalization level'],
    df_f_mult['Run']
))
df_f_mult = df_f_mult[~df_f_mult['Configuration'].isin(infeasible_runs)]
df_annual_res['Configuration'] = list(zip(
    df_annual_res['SSP-RCP'],
    df_annual_res['Regionalization level'],
    df_annual_res['Run']
))
df_annual_res = df_annual_res[~df_annual_res['Configuration'].isin(infeasible_runs)]
df_total_cost['Configuration'] = list(zip(
    df_total_cost['SSP-RCP'],
    df_total_cost['Regionalization level'],
    df_total_cost['Run']
))
df_total_cost = df_total_cost[~df_total_cost['Configuration'].isin(infeasible_runs)]

In [ ]:
if save_results:
    df_f_mult[df_f_mult.F_Mult != 0].to_csv(f'../03_Results/Tables/sensitivity_analysis/df_f_mult.csv', index=False)
    df_annual_prod[df_annual_prod.Annual_Prod != 0].to_csv(f'../03_Results/Tables/sensitivity_analysis/df_annual_prod.csv', index=False)
    df_annual_res[df_annual_res.Annual_Res != 0].to_csv(f'../03_Results/Tables/sensitivity_analysis/df_annual_res.csv', index=False)

In [ ]:
df_no_more_ng = df_annual_res[(df_annual_res['index'] == 'NG_EHP') & (df_annual_res['Annual_Res'] == 0)].groupby(['SSP-RCP', 'Regionalization level']).max()[['Run']]
df_low_ng = df_annual_res[(df_annual_res['index'] == 'NG_EHP') & (df_annual_res['Annual_Res'] <= 10)].groupby(['SSP-RCP', 'Regionalization level']).max()[['Run']]
no_more_ng_min = df_no_more_ng['Run'].min()
no_more_ng_max = df_no_more_ng['Run'].max()
low_ng_min = df_low_ng['Run'].min()
low_ng_max = df_low_ng['Run'].max()

In [ ]:
def plot_sensitivity(
        tech_or_res: str | list[str],
        type: str = 'tech',
        save_results: bool = save_results,
        show_legend: bool = False,
        show_fig: bool = True,
        file_name: str = None,
):
    if type == 'tech':
        df = df_annual_prod
    else:
        df = df_annual_res

    if isinstance(tech_or_res, str):
        tech_or_res = [tech_or_res]

    df = df[df['index'].isin(tech_or_res)].groupby(
        ["SSP-RCP", "Regionalization level", "Run"]
    ).sum().reset_index()

    y_col = "Annual_Prod" if type == 'tech' else 'Annual_Res'

    if type == 'tech':
        if tech_or_res[0].startswith(tuple(['LCV_', 'PLANE_FREIGHT_', 'TRAIN_FREIGHT_', 'SCHOOLBUS_', 'TRUCK_', 'SEMI_'])):
            y_label_name = "Freight mobility (Gtkm/year)"
        elif tech_or_res[0].startswith(tuple(['BUS_', 'COACH_', 'PLANE_', 'TRAIN_', 'SCHOOLBUS_', 'CAR_', 'SUV_'])):
            y_label_name = "Passenger mobility (Gpkm/year)"
        else:
            y_label_name = "Annual production (TWh/year)"
    else:
        y_label_name = "Imports (TWh/year)"

    fig = go.Figure()

    for (ssp, reg), grp in df.groupby(["SSP-RCP", "Regionalization level"]):
        if reg in ['base_wo_iam', 'spat_fore_back']:
            grp_sorted = grp.sort_values("Run")
            fig.add_trace(go.Scatter(
                x=grp_sorted["Run"],
                y=grp_sorted[y_col],
                mode="lines",
                line=dict(
                    color=c[ssp],#reg_colors[reg],
                    dash='solid', #ssp_ls[ssp],
                    width=2.5 if reg in ['base_wo_iam', 'spat_fore_back'] else 1.6,
                ),
                opacity=0.85,
                # name=f"{reg_level_name_dict_2050[reg]} - {ssp}" if ssp != 'base_wo_iam' else reg_level_name_dict_2050[reg],
                name=f"{ssp} ({reg_level_name_dict_2050[reg]})" if ssp != 'base_wo_iam' else 'Default',
                showlegend=show_legend,
                hovertemplate=(
                    f"<b>Prosp. reg level:</b> {f'{reg_level_name_dict_2050[reg]} - {ssp}' if ssp != 'base_wo_iam' else reg_level_name_dict_2050[reg]}<br>"
                    "<b>CCS avail.:</b> %{x} Mt CO<sub>2</sub>/year<br>"
                    f"<b>Value:</b> %{{y:.2f}} {y_label_name.split('(')[-1].replace(')','')}"
                    "<extra></extra>"
                ),
            ))

    for ssp_rcp in ['SSP2-L', 'SSP5-H']:
        df_area = df[df['SSP-RCP'] == ssp_rcp]
        grouped = df_area.groupby('Run')[y_col].agg(['min', 'max']).sort_index()

        x = list(grouped.index)
        x_rev = x[::-1]
        y_lower = list(grouped['min'])[::-1]
        y_upper = list(grouped['max'])

        fig.add_trace(go.Scatter(
            x=x+x_rev,
            y=y_upper+y_lower,
            fill='toself',
            showlegend=show_legend,
            name=f"{ssp_rcp} (other levels)",
            fillcolor=c[ssp_rcp],
            line=dict(width=0),
            opacity=0.2,
        ))

    y_max = df[y_col].max()

    fig.update_layout(
        width=480 if not show_legend else 800,
        height=432,
        xaxis=dict(
            title=dict(text="Carbon capture and storage availability<br>(Mt CO<sub>2</sub>/year)", font=dict(size=20), standoff=5),
            tickfont=dict(size=17),
            range=[0.8 * df['Run'].min(), df['Run'].max() * 1.02],
        ),
        yaxis=dict(
            title=dict(text=y_label_name, font=dict(size=20), standoff=5),
            tickfont=dict(size=17),
            range=[y_max * (-0.05), y_max * 1.05],
        ),
        plot_bgcolor="white",
        margin=dict(l=10, r=10, t=10, b=10),
    )

    # fig.add_vrect(
    #     x0=no_more_ng_min, x1=low_ng_min,
    #     fillcolor="yellow", opacity=0.2, line_width=0,
    #     name="All configs. ≤10 TWh NG imports",
    #     legendgroup="shading",
    #     legendgrouptitle_text="Shaded regions",
    #     showlegend=show_legend,
    # )
    # fig.add_vrect(
    #     x0=avail[0]['lower_bound'] / 1e3, x1=no_more_ng_min,
    #     fillcolor="green", opacity=0.1, line_width=0,
    #     name="All configs. no NG imports",
    #     legendgroup="shading",
    #     legendgrouptitle_text="Shaded regions",
    #     showlegend=show_legend,
    # )

    # fig.add_vline(
    #     x=low_ng_min,
    #     line_width=2.5, line_dash="dashdot", line_color="green",
    #     name="All configs. ≤10 TWh NG imports",
    #     legendgroup="shading",
    #     legendgrouptitle_text="Shaded regions",
    #     showlegend=show_legend,
    # )
    # fig.add_vline(
    #     x=no_more_ng_min,
    #     line_width=2.5, line_dash="dash", line_color="green",
    #     name="All configs. no NG imports",
    #     legendgroup="shading",
    #     legendgrouptitle_text="Shaded regions",
    #     showlegend=show_legend,
    # )

    if show_legend:
        # for label, col in reg_colors.items():
        #     if label == 'base_wo_iam':
        #         continue
        #     fig.add_trace(go.Scatter(
        #         x=[None], y=[None], mode="lines",
        #         line=dict(color=col, width=3 if label == 'spat_fore_back' else 2),
        #         name=reg_level_name_dict_2050[label],
        #         legendgroup="reg_level",
        #         legendgrouptitle_text="Regionalization level" if label == list(reg_colors.keys())[0] else None,
        #         showlegend=True,
        #     ))
        #
        # for ssp, ls in ssp_ls.items():
        #     label = 'Def.' if ssp == 'base_wo_iam' else ssp
        #     fig.add_trace(go.Scatter(
        #         x=[None], y=[None], mode="lines",
        #         line=dict(color="black", dash=ls, width=3 if ssp == 'base_wo_iam' else 2),
        #         name=label,
        #         legendgroup="ssp_rcp",
        #         legendgrouptitle_text="SSP-RCP" if ssp == list(ssp_ls.keys())[0] else None,
        #         showlegend=True,
        #     ))
        #
        # fig.add_trace(go.Scatter(
        #     x=[None], y=[None], mode="markers",
        #     marker=dict(symbol="square", size=14, color="yellow", opacity=0.2),
        #     name="All configs. ≤10 TWh NG imports",
        #     legendgroup="shading",
        #     legendgrouptitle_text="Shaded regions",
        #     showlegend=True,
        # ))
        # fig.add_trace(go.Scatter(
        #     x=[None], y=[None], mode="markers",
        #     marker=dict(symbol="square", size=14, color="green", opacity=0.1),
        #     name="All configs. no NG imports",
        #     legendgroup="shading",
        #     showlegend=True,
        # ))

        fig.update_layout(
            legend=dict(
                orientation="v",
                yanchor="top", y=1,
                xanchor="left", x=1.02,
                font=dict(size=13),
                groupclick="toggleitem",
                title_text="Prospective-regionalization modeling level",
            )
        )
    # else:
    #     fig.update_layout(showlegend=False)

    fig.update_xaxes(dtick=5)

    if file_name is None:
        file_name = tech_or_res[0].lower()

    if show_fig:
        fig.show()

    if save_results:
        # fig.write_image(
        #     f'../03_Results/Figures/sensitivity_analysis/carbon_seq_avail_{file_name}.pdf'
        # )
        fig.update_layout(width=None, height=None)
        fig.write_html(
            f'../03_Results/Figures/sensitivity_analysis/carbon_seq_avail_{file_name}.html',
            include_plotlyjs=True,
            full_html=True,
        )

In [ ]:
df_changes = pd.merge(
    df_annual_prod,
    df_annual_prod[df_annual_prod['Run'] == float(df_annual_prod['Run'].max())][['index', 'SSP-RCP', 'Regionalization level', 'Annual_Prod']],
    on=['index', 'SSP-RCP', 'Regionalization level'],
    suffixes=('', '_N_max'),
)[['index', 'SSP-RCP', 'Regionalization level', 'Annual_Prod', 'Annual_Prod_N_max', 'Run']]

df_changes['Delta'] = abs(df_changes['Annual_Prod_N_max'] - df_changes['Annual_Prod'])

In [ ]:
list_most_impacted_technos = df_changes[~df_changes['index'].str.contains('|'.join(['GRID', 'TRAFO', '_EXP_', '_COMP_', '_SD', '_MD', '_LD', '_ELD']))][['index', 'Delta']].groupby('index').sum().sort_values('Delta', ascending=False).head(30).index.tolist()

In [ ]:
list_most_impacted_technos

In [ ]:
model = pd.read_csv(f'../01_Notebooks/Data/model_{year}.csv')
h2_prod_techs = model[(model.Flow.isin(['H2_LP', 'H2_MP', 'H2_HP', 'H2_EHP'])) & (model.Amount == 1.0) & (~model.Name.str.contains('|'.join(['_ST0', 'STO_', '_EXP_', '_COMP_'])))].Name.tolist()
sng_prod_techs = model[(model.Flow.isin(['SNG_LP', 'SNG_MP', 'SNG_HP', 'SNG_EHP'])) & (model.Amount == 1.0) & (~model.Name.str.contains('|'.join(['_ST0', 'STO_', '_EXP_', '_COMP_'])))].Name.tolist()

In [ ]:
show_legend = True

In [ ]:
plot_sensitivity(sng_prod_techs, file_name='sng', show_legend=show_legend)

In [ ]:
plot_sensitivity(h2_prod_techs, file_name='h2', show_legend=show_legend)

In [ ]:
plot_sensitivity(['ALKALINE_ELECTROLYSIS', 'PEM_ELECTROLYSIS', 'SOEC_ELECTROLYSIS'], show_legend=True, file_name='electrolysis')

In [ ]:
plot_sensitivity('NEW_WIND_ONSHORE', show_legend=show_legend)

In [ ]:
df_changes = df_changes[df_changes.Annual_Prod_N_max != 0]
df_changes['Delta_rel (%)'] = 100 * (df_changes['Annual_Prod'] - df_changes['Annual_Prod_N_max']) / df_changes['Annual_Prod_N_max']

In [ ]:
df_changes[(df_changes['index'] == 'NEW_WIND_ONSHORE') & (df_changes['Run'] == avail_min)]

In [ ]:
df_f_mult[(df_f_mult['index'] == 'NEW_WIND_ONSHORE') & (df_f_mult.Run == avail_min)]

In [ ]:
for tec in [
    'NEW_HYDRO_DAM',
    'NEW_WIND_ONSHORE',
    'IND_DIRECT_ELEC',
    'IND_BOILER_GAS',
    'IND_BOILER_WOOD',
    'SEMI_LH_CNG',
    'SEMI_LH_FC_H2',
    'GASIFICATION_H2',
    'SUV_FC_H2',
    'SUV_EV',
    'SEMI_SH_BIODIESEL_B100',
    'SEMI_SH_HY_DIESEL',
    'SMR',
    'DEC_HP_ELEC',
    'DEC_BOILER_GAS',
    'NG_EHP',
]:
    plot_sensitivity(tec, show_fig=False, show_legend=show_legend)

In [ ]:
plot_sensitivity('NG_EHP', type='res', show_legend=show_legend)

In [ ]:
plot_sensitivity('METHANOL', type='res', show_legend=show_legend)

In [ ]:
plot_sensitivity('BIO_ETHANOL', type='res', show_legend=show_legend)

In [ ]:
plot_sensitivity(
    [
        'BIOMASS_AGRICULTURE_DEJECTION', 'BIOMASS_AGRICULTURE_RESIDUAL',
        'BIOMASS_FORESTRY_LEFTOVER1', 'BIOMASS_FORESTRY_LEFTOVER2',
        'BIOMASS_FORESTRY_RESIDUAL', 'BIOMASS_FORESTRY_UNEXPLOITED',
        'BIOMASS_WASTE_BUILDING', 'BIOMASS_WASTE_MUNICIPAL',
        'BIOMASS_WASTE_ORGANIC', 'BIOMASS_WASTE_PAPER',
        'BIOMASS_WASTE_PAPER_FABRIC',
    ],
    type='res',
    file_name='biomass',
    show_legend=show_legend,
)

In [ ]:
df_cost_ori = pd.read_csv('../03_Results/Tables/2050/df_results_constraints.csv')

In [ ]:
reg_level_name_dict_2050_rev = {v: k for k, v in reg_level_name_dict_2050.items()}

In [ ]:
df_cost_ori['Run'] = df_cost_ori['Run'].apply(lambda x: reg_level_name_dict_2050_rev[x])
df_cost_ori['Regionalization level'] = df_cost_ori.apply(lambda x: x['Run'].split('-')[0], axis=1)
df_cost_ori['SSP-RCP'] = df_cost_ori.apply(lambda x: "-".join(x['Run'].split('-')[1:]) if x['Regionalization level'] != 'base_wo_iam' else 'base_wo_iam', axis=1)

In [ ]:
df_total_cost = df_total_cost.merge(
    # df_total_cost.groupby(['SSP-RCP', 'Regionalization level']).min()['TotalCost'].reset_index().rename(columns={'TotalCost': 'min_TotalCost'}),
    df_cost_ori[['SSP-RCP', 'Regionalization level', 'Total cost']].rename(columns={'Total cost': 'min_TotalCost'}),
    on=['SSP-RCP', 'Regionalization level'],
    how='left',
)
df_total_cost['Additional cost (%)'] = 100 * (df_total_cost['TotalCost'] - df_total_cost['min_TotalCost']) / df_total_cost['min_TotalCost']

In [ ]:
if save_results:
    df_total_cost.to_csv(f'../03_Results/Tables/sensitivity_analysis/df_total_cost.csv', index=False)

In [ ]:
fig = go.Figure()

for (ssp, reg), grp in df_total_cost.groupby(["SSP-RCP", "Regionalization level"]):
    if reg in ['base_wo_iam', 'spat_fore_back']:
        grp_sorted = grp.sort_values("Run")
        fig.add_trace(go.Scatter(
            x=grp_sorted["Run"],
            y=grp_sorted["Additional cost (%)"],
            mode="lines",
            line=dict(
                color=c[ssp], #reg_colors[reg],
                # dash=ssp_ls[ssp],
                width=2.6 if reg in ['base_wo_iam', 'spat_fore_back'] else 1.6,
            ),
            opacity=0.85,
            name=f'{reg_level_name_dict_2050[reg]} - {ssp}' if ssp != 'base_wo_iam' else reg_level_name_dict_2050[reg],
            showlegend=show_legend,
            hovertemplate=(
                f"<b>Prosp. reg level:</b> {f'{reg_level_name_dict_2050[reg]} - {ssp}' if ssp != 'base_wo_iam' else reg_level_name_dict_2050[reg]}<br>"
                "<b>CCS avail.:</b> %{x} Mt CO<sub>2</sub>/year<br>"
                "<b>Value:</b> %{y:.2f}%"
                "<extra></extra>"
            ),
        ))

for ssp_rcp in ['SSP2-L', 'SSP5-H']:
    df_area = df_total_cost[df_total_cost['SSP-RCP'] == ssp_rcp]
    grouped = df_area.groupby('Run')['Additional cost (%)'].agg(['min', 'max']).sort_index()

    x = list(grouped.index)
    x_rev = x[::-1]
    y_lower = list(grouped['min'])[::-1]
    y_upper = list(grouped['max'])

    fig.add_trace(go.Scatter(
        x=x+x_rev,
        y=y_upper+y_lower,
        fill='toself',
        showlegend=show_legend,
        name=f"{ssp_rcp} (other levels)",
        fillcolor=c[ssp_rcp],
        line=dict(width=0),
        opacity=0.2,
    ))

fig.update_layout(
    width=480 if not show_legend else 700,
    height=432,
    xaxis=dict(
        title=dict(text="Carbon capture and storage availability<br>(Mt CO<sub>2</sub>/year)", font=dict(size=20), standoff=5),
        tickfont=dict(size=17),
        range=[0.8 * df_total_cost['Run'].min(), df_total_cost['Run'].max() * 1.02],
    ),
    yaxis=dict(
        title=dict(text="Additional cost (%)", font=dict(size=20), standoff=5),
        tickfont=dict(size=17),
    ),
    plot_bgcolor="white",
    margin=dict(l=10, r=10, t=10, b=10),
)

# fig.add_vrect(
#     x0=no_more_ng_min, x1=low_ng_min,
#     fillcolor="yellow", opacity=0.2, line_width=0,
#     name="All configs. ≤10 TWh NG imports",
#     legendgroup="shading",
#     legendgrouptitle_text="Shaded regions",
#     showlegend=show_legend,
# )
# fig.add_vrect(
#     x0=avail[0]['lower_bound'] / 1e3, x1=no_more_ng_min,
#     fillcolor="green", opacity=0.1, line_width=0,
#     name="All configs. no NG imports",
#     legendgroup="shading",
#     legendgrouptitle_text="Shaded regions",
#     showlegend=show_legend,
# )

fig.update_xaxes(dtick=5)

if show_legend:
    fig.update_layout(
        legend=dict(
            orientation="v",
            yanchor="top", y=1,
            xanchor="left", x=1.02,
            font=dict(size=13),
            groupclick="toggleitem",
            title_text="Prospective-regionalization modeling level",
        )
    )

fig.show()

if save_results:
    # fig.write_image(
    #     f'../03_Results/Figures/sensitivity_analysis/carbon_seq_avail_cost.pdf'
    # )
    fig.update_layout(width=None, height=None)
    fig.write_html(
        f'../03_Results/Figures/sensitivity_analysis/carbon_seq_avail_cost.html',
        include_plotlyjs=True,
        full_html=True,
    )

In [ ]:
df_total_cost.groupby(['SSP-RCP', 'Regionalization level'])[['Additional cost (%)']].max()

In [ ]:
cat_list = ['Total human health (biogenic)', 'Total ecosystem quality (biogenic)', 'Climate change, short term, total', 'Remaining human health', 'Remaining ecosystem quality']
col = ['Run', 'index', 'SSP-RCP', 'Regionalization level', 'Phase', 'Sector'] + cat_list
df_f_mult['Phase'] = 'Construction'
df_annual_prod['Phase'] = 'Operation'
df_annual_res['Phase'] = 'Resource'
df_total_impact = pd.concat([
    df_f_mult[col+['F_Mult']].rename(columns={'F_Mult': 'Production, capacity or import'}),
    df_annual_prod[col+['Annual_Prod']].rename(columns={'Annual_Prod': 'Production, capacity or import'}),
    df_annual_res[col+['Annual_Res']].rename(columns={'Annual_Res': 'Production, capacity or import'}),
],
    ignore_index=True)

In [ ]:
df_total_impact['Total human health (biogenic)'] *= 1e6 / N_capita_2050
df_total_impact['Total ecosystem quality (biogenic)'] *= 1e6 / N_capita_2050
df_total_impact['Remaining human health'] *= 1e6 / N_capita_2050
df_total_impact['Remaining ecosystem quality'] *= 1e6 / N_capita_2050
df_total_impact['Climate change, short term, total'] *= 1e3 / N_capita_2050

In [ ]:
all_mob_techs = (
    list(res.sets['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES'])
    + list(res.sets['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES'])
    + list(res.sets['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES'])
)

# Keeping mobility sub-models only
df_total_impact = df_total_impact[~df_total_impact['index'].isin(all_mob_techs)]

In [ ]:
if save_results:
    df_total_impact[df_total_impact['Production, capacity or import'] != 0].to_csv(f'../03_Results/Tables/sensitivity_analysis/df_total_impact.csv', index=False)

In [ ]:
df_total_impact = df_total_impact.groupby(["SSP-RCP", "Regionalization level", "Run"]).sum()[cat_list].reset_index()

In [ ]:
df_total_impact = pd.merge(
    df_total_impact,
    df_total_impact[df_total_impact['Run'] == float(df_total_impact['Run'].max())],
    on = ['SSP-RCP', 'Regionalization level'],
    suffixes=['', ' (N_max)'],
)

for cat in cat_list:
    df_total_impact[f"{cat} delta_rel (%)"] = 100 * (df_total_impact[cat] - df_total_impact[f"{cat} (N_max)"]) / df_total_impact[f"{cat} (N_max)"]

In [ ]:
df_total_impact[df_total_impact.Run == avail_min]

In [ ]:
def plot_sensitivity_impact(imp_cat, show_legend: bool = False):

    if imp_cat == 'Total human health (biogenic)':
        suffix = 'tthh'
    elif imp_cat == 'Remaining human health':
        suffix = 'rhh'
    elif imp_cat == 'Total ecosystem quality (biogenic)':
        suffix = 'tteq'
    elif imp_cat == 'Remaining ecosystem quality':
        suffix = 'req'
    elif imp_cat == 'Climate change, short term, total':
        suffix = 'ccst'

    y_max = df_total_impact[imp_cat].max()
    y_label = f'{imp_cat.replace(" (biogenic)", "").replace(", total", "")}<br>({unit_dict_plotly[imp_cat.replace(" (biogenic)", "").replace(", total", "")]})'

    fig = go.Figure()

    for (ssp, reg), grp in df_total_impact.groupby(["SSP-RCP", "Regionalization level"]):
        if reg in ['base_wo_iam', 'spat_fore_back']:
            grp_sorted = grp.sort_values("Run")
            fig.add_trace(go.Scatter(
                x=grp_sorted["Run"],
                y=grp_sorted[imp_cat],
                mode="lines",
                line=dict(
                    color=c[ssp],#reg_colors[reg],
                    dash='solid', #ssp_ls[ssp],
                    width=2.5 if reg in ['base_wo_iam', 'spat_fore_back'] else 1.6,
                ),
                opacity=0.85,
                # name=f"{reg_level_name_dict_2050[reg]} - {ssp}" if ssp != 'base_wo_iam' else reg_level_name_dict_2050[reg],
                name=f"{ssp} ({reg_level_name_dict_2050[reg]})" if ssp != 'base_wo_iam' else 'Default',
                showlegend=show_legend,
                hovertemplate=(
                f"<b>Prosp. reg level:</b> {f'{reg_level_name_dict_2050[reg]} - {ssp}' if ssp != 'base_wo_iam' else reg_level_name_dict_2050[reg]}<br>"
                "<b>CCS avail.:</b> %{x} Mt CO<sub>2</sub>/year<br>"
                f"<b>Value:</b> %{{y:.2e}} {unit_dict_plotly[imp_cat.replace(' (biogenic)', '').replace(', total', '')]}"
                "<extra></extra>"),
            ))

    for ssp_rcp in ['SSP2-L', 'SSP5-H']:
        df_area = df_total_impact[df_total_impact['SSP-RCP'] == ssp_rcp]
        grouped = df_area.groupby('Run')[imp_cat].agg(['min', 'max']).sort_index()

        x = list(grouped.index)
        x_rev = x[::-1]
        y_lower = list(grouped['min'])[::-1]
        y_upper = list(grouped['max'])

        fig.add_trace(go.Scatter(
            x=x+x_rev,
            y=y_upper+y_lower,
            fill='toself',
            showlegend=show_legend,
            name=f"{ssp_rcp} (other levels)",
            fillcolor=c[ssp_rcp],
            line=dict(width=0),
            opacity=0.2,
        ))

    fig.update_layout(
        width=547 if not show_legend else 700,
        height=432,
        xaxis=dict(
            title=dict(text="Carbon capture and storage availability<br>(Mt CO<sub>2</sub>/year)", font=dict(size=20), standoff=5),
            tickfont=dict(size=17),
            range=[0.8 * df_total_impact['Run'].min(), df_total_impact['Run'].max() * 1.02],
        ),
        yaxis=dict(
            title=dict(text=y_label, font=dict(size=20), standoff=5),
            tickfont=dict(size=17),
            range=[y_max * (-0.05), y_max * 1.05],
        ),
        plot_bgcolor="white",
        margin=dict(l=10, r=10, t=10, b=10),
    )

    fig.update_xaxes(dtick=5)

    # fig.add_vrect(
    #     x0=no_more_ng_min, x1=low_ng_min,
    #     fillcolor="yellow", opacity=0.2, line_width=0,
    #     name="All configs. ≤10 TWh NG imports",
    #     legendgroup="shading",
    #     legendgrouptitle_text="Shaded regions",
    #     showlegend=show_legend,
    # )
    # fig.add_vrect(
    #     x0=avail[0]['lower_bound'] / 1e3, x1=no_more_ng_min,
    #     fillcolor="green", opacity=0.1, line_width=0,
    #     name="All configs. no NG imports",
    #     legendgroup="shading",
    #     legendgrouptitle_text="Shaded regions",
    #     showlegend=show_legend,
    # )
    #
    if show_legend:
        fig.update_layout(
            legend=dict(
                yanchor="top",
                y=1,
                xanchor="left",
                x=1.02,
                title_text="Prospective-regionalization modeling level",
            )
        )

    fig.show()

    if save_results:
        # fig.write_image(
        #     f'../03_Results/Figures/sensitivity_analysis/carbon_seq_avail_{suffix}.pdf'
        # )
        fig.update_layout(width=None, height=None)
        fig.write_html(
            f'../03_Results/Figures/sensitivity_analysis/carbon_seq_avail_{suffix}.html',
            include_plotlyjs=True,
            full_html=True,
        )

In [ ]:
plot_sensitivity_impact('Remaining ecosystem quality', show_legend=show_legend)

In [ ]:
plot_sensitivity_impact('Remaining human health', show_legend=show_legend)

In [ ]:
plot_sensitivity_impact('Total human health (biogenic)', show_legend=show_legend)

In [ ]:
plot_sensitivity_impact('Total ecosystem quality (biogenic)', show_legend=show_legend)

In [ ]:
plot_sensitivity_impact('Climate change, short term, total', show_legend=show_legend)